# 💰 Project 01 — Expense Tracker (OOP + Files + Exceptions + Reports)

**Skills used:** classes, dataclasses, `datetime`, dictionaries, comprehensions, JSON persistence, custom exceptions, `collections`, string formatting, a command loop.

**What you'll build:** a tracker that records expenses, saves them to a JSON file, and prints summaries by category and month, with budget warnings.

**Suggested workflow:** read each step, type it yourself in a fresh notebook, run it, then extend it with the challenges at the end.

---
## Step 1 — Model the data (`dataclass`)

In [1]:
from dataclasses import dataclass, asdict
from datetime import date, datetime

@dataclass
class Expense:
    amount: float
    category: str
    description: str = ""
    day: str = ""                # ISO date "YYYY-MM-DD"

    def __post_init__(self):
        if self.amount <= 0:
            raise ValueError("amount must be positive")
        self.category = self.category.strip().lower()
        if not self.day:
            self.day = date.today().isoformat()
        datetime.strptime(self.day, "%Y-%m-%d")     # raises ValueError for bad dates

e = Expense(250, "Food", "lunch", "2025-03-14")
print(e)

Expense(amount=250, category='food', description='lunch', day='2025-03-14')


## Step 2 — The tracker class (add, remove, save, load)

In [2]:
import json, os
from collections import defaultdict

class ExpenseError(Exception):
    """Base class for tracker errors."""

class BudgetExceeded(ExpenseError):
    def __init__(self, category, spent, limit):
        super().__init__(f"Budget for '{category}' exceeded: {spent:.2f} / {limit:.2f}")
        self.category, self.spent, self.limit = category, spent, limit

class ExpenseTracker:
    def __init__(self, path="expenses.json"):
        self.path = path
        self.expenses: list[Expense] = []
        self.budgets: dict[str, float] = {}
        self.load()

    # ---- persistence ----
    def save(self):
        data = {"expenses": [asdict(e) for e in self.expenses], "budgets": self.budgets}
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

    def load(self):
        try:
            with open(self.path, encoding="utf-8") as f:
                data = json.load(f)
            self.expenses = [Expense(**item) for item in data["expenses"]]
            self.budgets = data.get("budgets", {})
        except FileNotFoundError:
            pass                                      # first run: start empty
        except (json.JSONDecodeError, KeyError) as e:
            raise ExpenseError(f"Corrupt data file: {e}") from e

    # ---- commands ----
    def add(self, amount, category, description="", day=""):
        exp = Expense(float(amount), category, description, day)
        self.expenses.append(exp)
        self.save()
        limit = self.budgets.get(exp.category)
        if limit is not None:
            spent = self.total(category=exp.category)
            if spent > limit:
                raise BudgetExceeded(exp.category, spent, limit)      # after saving: the expense IS recorded
        return exp

    def remove(self, index):
        try:
            removed = self.expenses.pop(index)
        except IndexError:
            raise ExpenseError(f"No expense #{index}") from None
        self.save()
        return removed

    def set_budget(self, category, limit):
        self.budgets[category.lower()] = float(limit)
        self.save()

    # ---- queries ----
    def total(self, category=None, month=None):
        return sum(e.amount for e in self._filter(category, month))

    def _filter(self, category=None, month=None):
        for e in self.expenses:
            if category and e.category != category.lower():
                continue
            if month and not e.day.startswith(month):          # month = "YYYY-MM"
                continue
            yield e

    def by_category(self, month=None):
        totals = defaultdict(float)
        for e in self._filter(month=month):
            totals[e.category] += e.amount
        return dict(sorted(totals.items(), key=lambda kv: -kv[1]))

    def by_month(self):
        totals = defaultdict(float)
        for e in self.expenses:
            totals[e.day[:7]] += e.amount
        return dict(sorted(totals.items()))

    def top(self, n=3):
        return sorted(self.expenses, key=lambda e: -e.amount)[:n]

    # ---- report ----
    def report(self, month=None):
        rows = self.by_category(month)
        grand = sum(rows.values()) or 1
        lines = [f"{'Category':<12}{'Amount':>10}{'Share':>8}  Bar", "-" * 44]
        for cat, amt in rows.items():
            share = amt / grand * 100
            lines.append(f"{cat:<12}{amt:>10,.2f}{share:>7.1f}%  {'█' * int(share // 5)}")
        lines += ["-" * 44, f"{'TOTAL':<12}{sum(rows.values()):>10,.2f}"]
        return "\n".join(lines)

## Step 3 — Try it out

In [3]:
import tempfile
tmp = os.path.join(tempfile.mkdtemp(), "demo_expenses.json")
tracker = ExpenseTracker(tmp)
tracker.set_budget("food", 500)

samples = [(250, "Food", "lunch", "2025-03-01"), (1200, "Rent", "March rent", "2025-03-02"),
           (180, "Transport", "metro pass", "2025-03-05"), (320, "Food", "groceries", "2025-03-09"),
           (999, "Fun", "concert", "2025-03-15"), (75, "Food", "snacks", "2025-04-02"),
           (1200, "Rent", "April rent", "2025-04-02")]
for amount, cat, desc, day in samples:
    try:
        tracker.add(amount, cat, desc, day)
    except BudgetExceeded as warn:
        print("⚠️ ", warn)

print(tracker.report("2025-03"))
print()
print("Monthly totals:", tracker.by_month())
print("Top 3:", [(e.description, e.amount) for e in tracker.top(3)])

⚠️  Budget for 'food' exceeded: 570.00 / 500.00
⚠️  Budget for 'food' exceeded: 645.00 / 500.00
Category        Amount   Share  Bar
--------------------------------------------
rent          1,200.00   40.7%  ████████
fun             999.00   33.9%  ██████
food            570.00   19.3%  ███
transport       180.00    6.1%  █
--------------------------------------------
TOTAL         2,949.00

Monthly totals: {'2025-03': 2949.0, '2025-04': 1275.0}
Top 3: [('March rent', 1200.0), ('April rent', 1200.0), ('concert', 999.0)]


## Step 4 — Persistence and error handling checks

In [4]:
# Reload from disk: the data survives
again = ExpenseTracker(tmp)
print(len(again.expenses), again.budgets)

# Bad input is rejected with clear errors
for bad in [(-5, "food"), (10, "food", "", "2025-13-45")]:
    try:
        again.add(*bad)
    except ValueError as e:
        print("Rejected:", bad, "->", e)

try:
    again.remove(99)
except ExpenseError as e:
    print("Error:", e)

# Corrupt file
with open(tmp, "w") as f:
    f.write("{not json")
try:
    ExpenseTracker(tmp)
except ExpenseError as e:
    print("Error:", e)

7 {'food': 500.0}
Rejected: (-5, 'food') -> amount must be positive
Rejected: (10, 'food', '', '2025-13-45') -> time data '2025-13-45' does not match format '%Y-%m-%d'
Error: No expense #99
Error: Corrupt data file: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)


## Step 5 — A command-line loop (interactive part)
The same class powers a menu. In a terminal this is a real app; here the cell is marked so it doesn't wait for typing.

In [ ]:
# notest
def main(path="expenses.json"):
    tracker = ExpenseTracker(path)
    menu = """
    1) Add expense   2) Report   3) Monthly totals   4) Remove   5) Set budget   q) Quit
    """
    while True:
        print(menu)
        choice = input("> ").strip().lower()
        try:
            if choice == "1":
                amount = float(input("Amount: "))
                category = input("Category: ")
                desc = input("Description: ")
                tracker.add(amount, category, desc)
                print("Saved ✔")
            elif choice == "2":
                print(tracker.report(input("Month YYYY-MM (blank = all): ") or None))
            elif choice == "3":
                for month, total in tracker.by_month().items():
                    print(month, f"{total:,.2f}")
            elif choice == "4":
                tracker.remove(int(input("Index: ")))
            elif choice == "5":
                tracker.set_budget(input("Category: "), float(input("Limit: ")))
            elif choice == "q":
                break
        except BudgetExceeded as warn:
            print("⚠️ ", warn)
        except (ValueError, ExpenseError) as err:
            print("❌", err)

# main()     # <- uncomment in a script: python expense_tracker.py

## 🧪 Add tests (habit of professionals)

In [5]:
import unittest

class TestTracker(unittest.TestCase):
    def setUp(self):
        self.path = os.path.join(tempfile.mkdtemp(), "t.json")
        self.t = ExpenseTracker(self.path)
        self.t.add(100, "food", day="2025-01-01")
        self.t.add(50, "food", day="2025-02-01")
        self.t.add(30, "fun", day="2025-01-15")

    def test_totals(self):
        self.assertEqual(self.t.total(), 180)
        self.assertEqual(self.t.total(category="food"), 150)
        self.assertEqual(self.t.total(month="2025-01"), 130)

    def test_by_category_sorted(self):
        self.assertEqual(list(self.t.by_category()), ["food", "fun"])

    def test_negative_rejected(self):
        with self.assertRaises(ValueError):
            self.t.add(-1, "food")

    def test_persistence(self):
        self.assertEqual(len(ExpenseTracker(self.path).expenses), 3)

    def test_budget(self):
        self.t.set_budget("fun", 40)
        with self.assertRaises(BudgetExceeded):
            self.t.add(20, "fun")

result = unittest.TextTestRunner(verbosity=0).run(unittest.defaultTestLoader.loadTestsFromTestCase(TestTracker))
print("all tests passed:", result.wasSuccessful())

all tests passed: True


----------------------------------------------------------------------
Ran 5 tests in 0.023s

OK


---
## 🚀 Extension challenges
1. Export the report to **CSV** and to an **HTML table**.
2. Add **recurring expenses** (rent every month) using `datetime` + `timedelta`.
3. Plot monthly totals with `matplotlib` (bar chart) and category share (pie chart).
4. Read/write with **SQLite** instead of JSON (see Module 12) — what changes?
5. Add **income** tracking and a "savings rate" metric.
6. Turn it into a small web app with **Streamlit** (`pip install streamlit`) — the class stays unchanged!

**What you practised:** separation of concerns (model / storage / UI), exception design, testing — exactly what interviewers look for.